### 도구(Tools)가 결합된 비주얼 에이전트

In [ ]:
%pip install -U ddgs

In [4]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
# Langchain에서 제공하는 무료 검색 툴
from langchain_community.tools import DuckDuckGoSearchRun

# 검색 도구 인스턴스 생성
search = DuckDuckGoSearchRun()

### 에이전트 로직 설계 (함수 분리)

In [5]:
# 1. 장소 식별 함수
def identify_landmark(image_url):
    """ 1단계 : 이미지 식별 - 사진을 보고 장소 이름을 반환 """

    model = ChatOpenAI(model="gpt-4o", temperature=0)

    prompt = "이 사진 속 장소의 정확한 이름만 딱 알려줘. 다른 말은 하지 말고."

    message = HumanMessage(
        content = [
            {"type":"text", "text":prompt},
            {"type":"image_url", "image_url":{"url":image_url}}
        ]
    )

    response = model.invoke([message])
    return response.content

In [6]:
# 2. 정보 검색 함수
def get_travel_info(keyword):
    """ 2단계 : 정보 검색 - 키워드로 인터넷 검색 수행"""
    print(f"검색어 : {keyword}")

    search_query = f"{keyword} travel guide interesting facts entry fee"

    result = search.invoke(search_query)
    return result

### 에이전트 통합

In [7]:
# 입력(이미지) 들어오면 -> 식별 -> 검색 -> 가이드 생성

def visual_agent(image_url):
    """ 통합 에이전트 함수 """

    print("이미지 분석 중")
    landmark_name = identify_landmark(image_url)
    print(f"식별된 장소 : {landmark_name}")

    print("정보 검색 중...")
    info = get_travel_info(landmark_name)

    final_model = ChatOpenAI(model='gpt-5-nano', temperature=0.7)

    prompt = f"""
    다음 정보를 바탕으로 '{landmark_name}'에 대한 친절한 여행 가이드 멘트를 한글로 작성해줘!

    [정보]
    {info}
    """

    summary = final_model.invoke(prompt)
    return summary.content

In [11]:
# test
image = "https://d1blyo8czty997.cloudfront.net/tour-photos/20537/800x800/8666094177.png"

result = visual_agent(image)
print(result)

이미지 분석 중
식별된 장소 : 피사의 사탑
정보 검색 중...
검색어 : 피사의 사탑
피사의 사탑 친절한 여행 가이드

- 왜 가야 하나
  피사의 사탑은 피사의 상징이자 직접 눈으로 확인해야만 느낄 수 있는 매력이 가득한 랜드마크입니다. 탑의 독특한 곡선과 광장은 사진 찍기에도, 걷기 산책하기에도 매력적인 공간이에요.

- 위치와 주변 환경
  사탑은 피사 대성당과 세례당이 함께 있는 미라콜리 광장(Piazza dei Miracoli) 안에 있어 광장 자체도 매우 아름답고 넓습니다. 광장에서는 축제나 다양한 공연·전시가 열리기도 하니 방문 시 꼭 한두 가지 이벤트를 확인해 보세요.

- 방문 정보 (필수 체크)
  - 내부로 올라가려면 계단을 걸어 올라가야 하며, 엘리베이터는 없습니다. 계단 수는 294개이고 왕복으로 최소 30분 정도 소요됩니다.
  - 8세 미만은 입장이 제한됩니다.
  - 사탑 내부를 올라가볼 계획이라면 예약이 필요할 수 있습니다. 특히 2025년 6월 30일 이후에는 시간대별 예약이 필요하다는 안내가 있습니다.
  - 사탑이나 광장을 그냥 사진으로만 남기고 싶다면 입장권은 필요 없이 미라콜리 광장 입장은 무료입니다.

- 사진 찍기 팁
  피사의 사탑은 사진 속에서 “들었다는 척” 혹은 “밀어내는 척”하는 포즈로 유명한 상징적인 사진 스팟이 많습니다. 탑의 끝없는 굽이진 모습과 주변 풍경을 배경으로 창의적인 연출이 가능하니 카메라를 들고 다양한 각도에서 도전해 보세요.

- 방문 코스 추천
  1) 미라콜리 광장을 먼저 산책하며 광장 분위기와 주변 건축물을 감상
  2) 탑으로 이동해 필요 시 예약 여부를 확인하고, 가능하면 내부 climbs 도전(시간대별 예약 필요 시 미리 준비)
  3) 인근 피사 대성당과 세례당도 함께 둘러보면 UNESCO 세계문화유산으로 등재된 이 지역의 가치를 한꺼번에 느낄 수 있습니다.
  4) 광장으로 돌아와 현지 공연이나 전시를 구경하거나 근처 맛집을 탐방

- UNESCO 세계문화유산
  피사의 사탑